# DSAA2011 Final Project: IBM AML Transaction Detection

This notebook documents the full project workflow: data preprocessing, visualization, clustering, supervised learning, evaluation, and graph-aware open-ended exploration.

## Dataset Summary

Full dataset rows: 5,078,345

Laundering rows: 5,177

Positive rate: 0.1019%

In [ ]:
from pathlib import Path

Path("src").mkdir(exist_ok=True)

## Full Reproducible Pipeline

The next cell writes the complete project pipeline to `src/aml_project_pipeline.py`. The pipeline downloads the dataset automatically if `data/HI-Small_Trans.csv.zip` is missing, then regenerates all figures and result tables.

To avoid re-running the full experiment every time you open this notebook, the final `main()` call is commented. Uncomment it when you want to reproduce every output from scratch.

In [ ]:
PIPELINE_SOURCE = 'from __future__ import annotations\n\nimport json\nimport math\nimport urllib.request\nimport zipfile\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport networkx as nx\nimport numpy as np\nimport pandas as pd\nimport seaborn as sns\nfrom sklearn.cluster import DBSCAN, MiniBatchKMeans\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.decomposition import PCA\nfrom sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.manifold import TSNE\nfrom sklearn.metrics import (\n    ConfusionMatrixDisplay,\n    average_precision_score,\n    calinski_harabasz_score,\n    classification_report,\n    confusion_matrix,\n    davies_bouldin_score,\n    f1_score,\n    precision_recall_curve,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n    roc_curve,\n    silhouette_score,\n)\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.tree import DecisionTreeClassifier\n\n\nROOT = Path(__file__).resolve().parents[1]\nDATA_ZIP = ROOT / "data" / "HI-Small_Trans.csv.zip"\nDATA_URL = (\n    "https://huggingface.co/datasets/eexzzm/"\n    "IBM-Transactions-for-Anti-Money-Laundering-HI-Small-Trans/resolve/main/"\n    "HI-Small_Trans.csv.zip?download=true"\n)\nFIGURES = ROOT / "figures"\nRESULTS = ROOT / "results"\nRANDOM_STATE = 42\n\n\ndef ensure_dirs() -> None:\n    for path in [ROOT / "data", FIGURES, RESULTS]:\n        path.mkdir(parents=True, exist_ok=True)\n\n\ndef ensure_data() -> None:\n    if DATA_ZIP.exists():\n        return\n    print(f"Downloading dataset to {DATA_ZIP}...")\n    urllib.request.urlretrieve(DATA_URL, DATA_ZIP)\n\n\ndef csv_name_in_zip() -> str:\n    with zipfile.ZipFile(DATA_ZIP) as zf:\n        names = [name for name in zf.namelist() if name.lower().endswith(".csv")]\n    if not names:\n        raise FileNotFoundError(f"No CSV file found in {DATA_ZIP}")\n    return names[0]\n\n\ndef first_pass_stats(chunksize: int = 500_000) -> dict:\n    total_rows = 0\n    positive_rows = 0\n    payment_counts = {}\n    payment_pos = {}\n    currency_counts = {}\n    currency_pos = {}\n    hourly_counts = {}\n    hourly_pos = {}\n    amount_by_label = {0: [], 1: []}\n\n    name = csv_name_in_zip()\n    with zipfile.ZipFile(DATA_ZIP) as zf:\n        for chunk in pd.read_csv(zf.open(name), chunksize=chunksize):\n            chunk["Timestamp"] = pd.to_datetime(chunk["Timestamp"], errors="coerce")\n            chunk["hour"] = chunk["Timestamp"].dt.hour\n            total_rows += len(chunk)\n            positive_rows += int(chunk["Is Laundering"].sum())\n\n            for col, count_map, pos_map in [\n                ("Payment Format", payment_counts, payment_pos),\n                ("Payment Currency", currency_counts, currency_pos),\n                ("hour", hourly_counts, hourly_pos),\n            ]:\n                counts = chunk.groupby(col, dropna=False).size()\n                poss = chunk.groupby(col, dropna=False)["Is Laundering"].sum()\n                for key, value in counts.items():\n                    count_map[str(key)] = count_map.get(str(key), 0) + int(value)\n                for key, value in poss.items():\n                    pos_map[str(key)] = pos_map.get(str(key), 0) + int(value)\n\n            for label in [0, 1]:\n                vals = chunk.loc[chunk["Is Laundering"] == label, "Amount Paid"].dropna()\n                if len(vals):\n                    amount_by_label[label].append(vals.sample(min(len(vals), 3000), random_state=RANDOM_STATE))\n\n    stats = {\n        "total_rows": total_rows,\n        "positive_rows": positive_rows,\n        "positive_rate": positive_rows / total_rows,\n        "payment_counts": payment_counts,\n        "payment_pos": payment_pos,\n        "currency_counts": currency_counts,\n        "currency_pos": currency_pos,\n        "hourly_counts": hourly_counts,\n        "hourly_pos": hourly_pos,\n    }\n    for label in [0, 1]:\n        if amount_by_label[label]:\n            sampled = pd.concat(amount_by_label[label], ignore_index=True)\n            stats[f"amount_label_{label}_sample"] = sampled.sample(\n                min(len(sampled), 50_000), random_state=RANDOM_STATE\n            ).tolist()\n    return stats\n\n\ndef plot_eda(stats: dict) -> None:\n    sns.set_theme(style="whitegrid", context="notebook")\n\n    with open(RESULTS / "dataset_stats.json", "w", encoding="utf-8") as f:\n        json.dump(stats, f, indent=2)\n\n    amount_df = pd.DataFrame(\n        {\n            "Amount Paid": stats.get("amount_label_0_sample", []) + stats.get("amount_label_1_sample", []),\n            "Label": ["Legitimate"] * len(stats.get("amount_label_0_sample", []))\n            + ["Laundering"] * len(stats.get("amount_label_1_sample", [])),\n        }\n    )\n    amount_df["log10_amount"] = np.log10(amount_df["Amount Paid"].clip(lower=1e-4))\n    plt.figure(figsize=(8, 5))\n    sns.histplot(data=amount_df, x="log10_amount", hue="Label", bins=60, stat="density", common_norm=False)\n    plt.title("Transaction Amount Distribution by Label")\n    plt.xlabel("log10(Amount Paid)")\n    plt.tight_layout()\n    plt.savefig(FIGURES / "eda_amount_distribution.png", dpi=220)\n    plt.close()\n\n    payment = pd.DataFrame(\n        [\n            {\n                "Payment Format": key,\n                "transactions": stats["payment_counts"][key],\n                "laundering": stats["payment_pos"].get(key, 0),\n            }\n            for key in stats["payment_counts"]\n        ]\n    )\n    payment["laundering_rate"] = payment["laundering"] / payment["transactions"]\n    payment = payment.sort_values("laundering_rate", ascending=False)\n    payment.to_csv(RESULTS / "payment_format_rates.csv", index=False)\n    plt.figure(figsize=(8, 5))\n    sns.barplot(data=payment, y="Payment Format", x="laundering_rate", color="#2E7D8F")\n    plt.title("Laundering Rate by Payment Format")\n    plt.xlabel("Laundering rate")\n    plt.ylabel("")\n    plt.tight_layout()\n    plt.savefig(FIGURES / "eda_payment_format_rate.png", dpi=220)\n    plt.close()\n\n    hourly = pd.DataFrame(\n        [\n            {\n                "hour": int(float(key)) if key != "nan" else -1,\n                "transactions": stats["hourly_counts"][key],\n                "laundering": stats["hourly_pos"].get(key, 0),\n            }\n            for key in stats["hourly_counts"]\n        ]\n    ).sort_values("hour")\n    hourly["laundering_rate"] = hourly["laundering"] / hourly["transactions"]\n    hourly.to_csv(RESULTS / "hourly_rates.csv", index=False)\n    fig, ax1 = plt.subplots(figsize=(9, 5))\n    ax1.bar(hourly["hour"], hourly["transactions"], color="#B7B7B7", label="Transactions")\n    ax1.set_ylabel("Transaction count")\n    ax2 = ax1.twinx()\n    ax2.plot(hourly["hour"], hourly["laundering_rate"], color="#C83E4D", marker="o", label="Laundering rate")\n    ax2.set_ylabel("Laundering rate")\n    ax1.set_xlabel("Hour of day")\n    plt.title("Transaction Volume and Laundering Rate by Hour")\n    fig.tight_layout()\n    plt.savefig(FIGURES / "eda_hourly_volume_rate.png", dpi=220)\n    plt.close()\n\n\ndef load_model_sample(stats: dict, target_negatives: int = 160_000, chunksize: int = 500_000) -> pd.DataFrame:\n    total_negatives = stats["total_rows"] - stats["positive_rows"]\n    neg_frac = min(1.0, target_negatives / total_negatives)\n    frames = []\n    name = csv_name_in_zip()\n    with zipfile.ZipFile(DATA_ZIP) as zf:\n        for chunk_index, chunk in enumerate(pd.read_csv(zf.open(name), chunksize=chunksize)):\n            positives = chunk[chunk["Is Laundering"] == 1]\n            negatives = chunk[chunk["Is Laundering"] == 0]\n            neg_sample = negatives.sample(frac=neg_frac, random_state=RANDOM_STATE + chunk_index)\n            frames.append(pd.concat([positives, neg_sample], ignore_index=True))\n    sample = pd.concat(frames, ignore_index=True)\n    sample = sample.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)\n    sample.to_csv(RESULTS / "model_sample_preview.csv", index=False)\n    return sample\n\n\ndef add_features(df: pd.DataFrame) -> pd.DataFrame:\n    df = df.copy()\n    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")\n    df["hour"] = df["Timestamp"].dt.hour.fillna(-1).astype(int)\n    df["weekday"] = df["Timestamp"].dt.weekday.fillna(-1).astype(int)\n    df["is_weekend"] = df["weekday"].isin([5, 6]).astype(int)\n    df["log_amount_paid"] = np.log1p(df["Amount Paid"].clip(lower=0))\n    df["log_amount_received"] = np.log1p(df["Amount Received"].clip(lower=0))\n    df["amount_diff"] = df["Amount Paid"] - df["Amount Received"]\n    df["same_currency"] = (df["Payment Currency"] == df["Receiving Currency"]).astype(int)\n    df["same_bank"] = (df["From Bank"] == df["To Bank"]).astype(int)\n    df["from_account_id"] = df["From Bank"].astype(str) + "_" + df["Account"].astype(str)\n    df["to_account_id"] = df["To Bank"].astype(str) + "_" + df["Account.1"].astype(str)\n    return df\n\n\ndef add_graph_features(df: pd.DataFrame) -> pd.DataFrame:\n    df = df.copy()\n    sent = df.groupby("from_account_id").agg(\n        sender_out_degree=("to_account_id", "size"),\n        sender_unique_receivers=("to_account_id", "nunique"),\n        sender_total_sent=("Amount Paid", "sum"),\n        sender_mean_sent=("Amount Paid", "mean"),\n    )\n    received = df.groupby("to_account_id").agg(\n        receiver_in_degree=("from_account_id", "size"),\n        receiver_unique_senders=("from_account_id", "nunique"),\n        receiver_total_received=("Amount Received", "sum"),\n        receiver_mean_received=("Amount Received", "mean"),\n    )\n    df = df.merge(sent, left_on="from_account_id", right_index=True, how="left")\n    df = df.merge(received, left_on="to_account_id", right_index=True, how="left")\n    df["sent_received_ratio"] = df["sender_total_sent"] / (df["receiver_total_received"] + 1e-6)\n\n    graph_sample = df.sample(min(len(df), 60_000), random_state=RANDOM_STATE)\n    graph = nx.from_pandas_edgelist(\n        graph_sample,\n        source="from_account_id",\n        target="to_account_id",\n        edge_attr=None,\n        create_using=nx.DiGraph(),\n    )\n    pagerank = nx.pagerank(graph, alpha=0.85, max_iter=80)\n    in_degree = dict(graph.in_degree())\n    out_degree = dict(graph.out_degree())\n    df["sender_pagerank"] = df["from_account_id"].map(pagerank).fillna(0.0)\n    df["receiver_pagerank"] = df["to_account_id"].map(pagerank).fillna(0.0)\n    df["sample_sender_out_degree"] = df["from_account_id"].map(out_degree).fillna(0.0)\n    df["sample_receiver_in_degree"] = df["to_account_id"].map(in_degree).fillna(0.0)\n    return df\n\n\nBASE_NUMERIC = [\n    "Amount Paid",\n    "Amount Received",\n    "log_amount_paid",\n    "log_amount_received",\n    "amount_diff",\n    "hour",\n    "weekday",\n    "is_weekend",\n    "same_currency",\n    "same_bank",\n]\n\nGRAPH_NUMERIC = [\n    "sender_out_degree",\n    "sender_unique_receivers",\n    "sender_total_sent",\n    "sender_mean_sent",\n    "receiver_in_degree",\n    "receiver_unique_senders",\n    "receiver_total_received",\n    "receiver_mean_received",\n    "sent_received_ratio",\n    "sender_pagerank",\n    "receiver_pagerank",\n    "sample_sender_out_degree",\n    "sample_receiver_in_degree",\n]\n\nCATEGORICAL = [\n    "Payment Currency",\n    "Receiving Currency",\n    "Payment Format",\n]\n\n\ndef make_preprocessor(numeric_features: list[str]) -> ColumnTransformer:\n    numeric_pipe = Pipeline(\n        steps=[\n            ("imputer", SimpleImputer(strategy="median")),\n            ("scaler", StandardScaler()),\n        ]\n    )\n    categorical_pipe = Pipeline(\n        steps=[\n            ("imputer", SimpleImputer(strategy="most_frequent")),\n            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False)),\n        ]\n    )\n    return ColumnTransformer(\n        transformers=[\n            ("num", numeric_pipe, numeric_features),\n            ("cat", categorical_pipe, CATEGORICAL),\n        ],\n        remainder="drop",\n    )\n\n\ndef run_clustering_and_tsne(df: pd.DataFrame) -> None:\n    pos = df[df["Is Laundering"] == 1]\n    neg = df[df["Is Laundering"] == 0].sample(min(8000, (df["Is Laundering"] == 0).sum()), random_state=RANDOM_STATE)\n    viz = pd.concat([pos, neg], ignore_index=True).sample(frac=1.0, random_state=RANDOM_STATE)\n    viz = viz.sample(min(len(viz), 12_000), random_state=RANDOM_STATE)\n\n    preprocessor = make_preprocessor(BASE_NUMERIC + GRAPH_NUMERIC)\n    x = preprocessor.fit_transform(viz)\n    pca_dims = min(20, x.shape[1], x.shape[0] - 1)\n    x_pca = PCA(n_components=pca_dims, random_state=RANDOM_STATE).fit_transform(x)\n\n    tsne = TSNE(\n        n_components=2,\n        perplexity=35,\n        init="pca",\n        learning_rate="auto",\n        max_iter=700,\n        random_state=RANDOM_STATE,\n    )\n    emb = tsne.fit_transform(x_pca)\n    viz["tsne_1"] = emb[:, 0]\n    viz["tsne_2"] = emb[:, 1]\n    viz.to_csv(RESULTS / "tsne_sample.csv", index=False)\n\n    plt.figure(figsize=(8, 6))\n    sns.scatterplot(\n        data=viz,\n        x="tsne_1",\n        y="tsne_2",\n        hue="Is Laundering",\n        palette={0: "#8A8F98", 1: "#C83E4D"},\n        s=9,\n        alpha=0.7,\n        linewidth=0,\n    )\n    plt.title("t-SNE Projection Colored by Laundering Label")\n    plt.tight_layout()\n    plt.savefig(FIGURES / "tsne_projection.png", dpi=220)\n    plt.close()\n\n    cluster_rows = []\n    kmeans = MiniBatchKMeans(n_clusters=6, random_state=RANDOM_STATE, batch_size=2048, n_init="auto")\n    viz["kmeans_cluster"] = kmeans.fit_predict(x_pca)\n    cluster_rows.append(evaluate_clusters("MiniBatchKMeans", x_pca, viz["kmeans_cluster"].to_numpy(), viz))\n\n    dbscan = DBSCAN(eps=2.8, min_samples=20)\n    viz["dbscan_cluster"] = dbscan.fit_predict(x_pca[:, : min(10, x_pca.shape[1])])\n    cluster_rows.append(evaluate_clusters("DBSCAN", x_pca, viz["dbscan_cluster"].to_numpy(), viz))\n\n    pd.DataFrame(cluster_rows).to_csv(RESULTS / "clustering_metrics.csv", index=False)\n\n    for col, fname, title in [\n        ("kmeans_cluster", "clustering_kmeans_tsne.png", "K-Means Clusters on t-SNE Projection"),\n        ("dbscan_cluster", "clustering_dbscan_tsne.png", "DBSCAN Clusters on t-SNE Projection"),\n    ]:\n        plt.figure(figsize=(8, 6))\n        sns.scatterplot(data=viz, x="tsne_1", y="tsne_2", hue=col, palette="tab10", s=9, alpha=0.75, linewidth=0)\n        plt.title(title)\n        plt.tight_layout()\n        plt.savefig(FIGURES / fname, dpi=220)\n        plt.close()\n\n    profiles = []\n    for method_col in ["kmeans_cluster", "dbscan_cluster"]:\n        prof = (\n            viz.groupby(method_col)\n            .agg(\n                transactions=("Is Laundering", "size"),\n                laundering=("Is Laundering", "sum"),\n                avg_amount=("Amount Paid", "mean"),\n                avg_sender_out_degree=("sender_out_degree", "mean"),\n            )\n            .reset_index()\n        )\n        prof["laundering_rate"] = prof["laundering"] / prof["transactions"]\n        prof["method"] = method_col\n        profiles.append(prof)\n    pd.concat(profiles, ignore_index=True).to_csv(RESULTS / "cluster_profiles.csv", index=False)\n\n\ndef evaluate_clusters(method: str, x: np.ndarray, labels: np.ndarray, viz: pd.DataFrame) -> dict:\n    unique = set(labels)\n    non_noise = labels != -1\n    valid = len(unique - {-1}) >= 2 and non_noise.sum() > 10\n    row = {\n        "method": method,\n        "n_clusters_ex_noise": len(unique - {-1}),\n        "noise_rate": float((labels == -1).mean()),\n        "silhouette": np.nan,\n        "davies_bouldin": np.nan,\n        "calinski_harabasz": np.nan,\n        "highest_cluster_laundering_rate": np.nan,\n    }\n    if valid:\n        used_x = x[non_noise] if -1 in unique else x\n        used_labels = labels[non_noise] if -1 in unique else labels\n        row["silhouette"] = float(silhouette_score(used_x, used_labels))\n        row["davies_bouldin"] = float(davies_bouldin_score(used_x, used_labels))\n        row["calinski_harabasz"] = float(calinski_harabasz_score(used_x, used_labels))\n    rates = viz.assign(cluster=labels).groupby("cluster")["Is Laundering"].mean()\n    if len(rates):\n        row["highest_cluster_laundering_rate"] = float(rates.max())\n    return row\n\n\ndef evaluate_classifier(name: str, model: Pipeline, x_train: pd.DataFrame, x_test: pd.DataFrame, y_train: pd.Series, y_test: pd.Series) -> dict:\n    model.fit(x_train, y_train)\n    if hasattr(model, "predict_proba"):\n        y_score = model.predict_proba(x_test)[:, 1]\n    else:\n        y_score = model.decision_function(x_test)\n    y_pred_default = (y_score >= 0.5).astype(int)\n\n    precision, recall, thresholds = precision_recall_curve(y_test, y_score)\n    f1_values = 2 * precision * recall / np.maximum(precision + recall, 1e-12)\n    best_idx = int(np.nanargmax(f1_values))\n    best_threshold = thresholds[max(best_idx - 1, 0)] if len(thresholds) else 0.5\n    y_pred_tuned = (y_score >= best_threshold).astype(int)\n\n    report = classification_report(y_test, y_pred_tuned, output_dict=True, zero_division=0)\n    row = {\n        "model": name,\n        "threshold": float(best_threshold),\n        "accuracy": float((y_pred_tuned == y_test).mean()),\n        "precision": float(precision_score(y_test, y_pred_tuned, zero_division=0)),\n        "recall": float(recall_score(y_test, y_pred_tuned, zero_division=0)),\n        "f1": float(f1_score(y_test, y_pred_tuned, zero_division=0)),\n        "roc_auc": float(roc_auc_score(y_test, y_score)),\n        "pr_auc": float(average_precision_score(y_test, y_score)),\n        "positive_precision": float(report["1"]["precision"]),\n        "positive_recall": float(report["1"]["recall"]),\n        "positive_f1": float(report["1"]["f1-score"]),\n    }\n\n    cm = confusion_matrix(y_test, y_pred_tuned)\n    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate", "Laundering"])\n    disp.plot(cmap="Blues", values_format="d")\n    plt.title(f"Confusion Matrix: {name}")\n    plt.tight_layout()\n    safe_name = name.lower().replace(" ", "_").replace("+", "plus")\n    plt.savefig(FIGURES / f"confusion_matrix_{safe_name}.png", dpi=220)\n    plt.close()\n\n    fpr, tpr, _ = roc_curve(y_test, y_score)\n    pr_precision, pr_recall, _ = precision_recall_curve(y_test, y_score)\n    pd.DataFrame({"fpr": fpr, "tpr": tpr}).to_csv(RESULTS / f"roc_{safe_name}.csv", index=False)\n    pd.DataFrame({"precision": pr_precision, "recall": pr_recall}).to_csv(RESULTS / f"pr_{safe_name}.csv", index=False)\n    return row\n\n\ndef run_supervised_models(df: pd.DataFrame) -> None:\n    y = df["Is Laundering"].astype(int)\n    x = df.drop(columns=["Is Laundering"])\n    x_train, x_test, y_train, y_test = train_test_split(\n        x, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y\n    )\n\n    base_pre = make_preprocessor(BASE_NUMERIC)\n    graph_pre = make_preprocessor(BASE_NUMERIC + GRAPH_NUMERIC)\n\n    models = [\n        (\n            "Logistic Regression",\n            Pipeline(\n                [\n                    ("preprocess", base_pre),\n                    (\n                        "model",\n                        LogisticRegression(\n                            max_iter=1000,\n                            class_weight="balanced",\n                            n_jobs=-1,\n                            random_state=RANDOM_STATE,\n                        ),\n                    ),\n                ]\n            ),\n        ),\n        (\n            "Decision Tree",\n            Pipeline(\n                [\n                    ("preprocess", base_pre),\n                    ("model", DecisionTreeClassifier(max_depth=9, class_weight="balanced", random_state=RANDOM_STATE)),\n                ]\n            ),\n        ),\n        (\n            "Random Forest + Graph Features",\n            Pipeline(\n                [\n                    ("preprocess", graph_pre),\n                    (\n                        "model",\n                        RandomForestClassifier(\n                            n_estimators=140,\n                            max_depth=16,\n                            min_samples_leaf=5,\n                            class_weight="balanced_subsample",\n                            n_jobs=-1,\n                            random_state=RANDOM_STATE,\n                        ),\n                    ),\n                ]\n            ),\n        ),\n        (\n            "Random Forest Base Features",\n            Pipeline(\n                [\n                    ("preprocess", make_preprocessor(BASE_NUMERIC)),\n                    (\n                        "model",\n                        RandomForestClassifier(\n                            n_estimators=140,\n                            max_depth=16,\n                            min_samples_leaf=5,\n                            class_weight="balanced_subsample",\n                            n_jobs=-1,\n                            random_state=RANDOM_STATE,\n                        ),\n                    ),\n                ]\n            ),\n        ),\n        (\n            "HistGradientBoosting Base Features",\n            Pipeline(\n                [\n                    ("preprocess", make_preprocessor(BASE_NUMERIC)),\n                    (\n                        "model",\n                        HistGradientBoostingClassifier(\n                            max_iter=120,\n                            learning_rate=0.08,\n                            max_leaf_nodes=31,\n                            random_state=RANDOM_STATE,\n                            class_weight="balanced",\n                        ),\n                    ),\n                ]\n            ),\n        ),\n        (\n            "HistGradientBoosting + Graph Features",\n            Pipeline(\n                [\n                    ("preprocess", make_preprocessor(BASE_NUMERIC + GRAPH_NUMERIC)),\n                    (\n                        "model",\n                        HistGradientBoostingClassifier(\n                            max_iter=120,\n                            learning_rate=0.08,\n                            max_leaf_nodes=31,\n                            random_state=RANDOM_STATE,\n                            class_weight="balanced",\n                        ),\n                    ),\n                ]\n            ),\n        ),\n    ]\n\n    rows = []\n    fitted = {}\n    for name, model in models:\n        print(f"Training {name}...")\n        rows.append(evaluate_classifier(name, model, x_train, x_test, y_train, y_test))\n        fitted[name] = model\n    metrics = pd.DataFrame(rows).sort_values(["positive_f1", "pr_auc"], ascending=False)\n    metrics.to_csv(RESULTS / "metrics_summary.csv", index=False)\n\n    plt.figure(figsize=(8, 6))\n    for _, row in metrics.iterrows():\n        safe_name = row["model"].lower().replace(" ", "_").replace("+", "plus")\n        curve = pd.read_csv(RESULTS / f"roc_{safe_name}.csv")\n        plt.plot(curve["fpr"], curve["tpr"], label=f"{row[\'model\']} AUC={row[\'roc_auc\']:.3f}")\n    plt.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)\n    plt.xlabel("False Positive Rate")\n    plt.ylabel("True Positive Rate")\n    plt.title("ROC Curves")\n    plt.legend(fontsize=8)\n    plt.tight_layout()\n    plt.savefig(FIGURES / "roc_curves.png", dpi=220)\n    plt.close()\n\n    plt.figure(figsize=(8, 6))\n    for _, row in metrics.iterrows():\n        safe_name = row["model"].lower().replace(" ", "_").replace("+", "plus")\n        curve = pd.read_csv(RESULTS / f"pr_{safe_name}.csv")\n        plt.plot(curve["recall"], curve["precision"], label=f"{row[\'model\']} AP={row[\'pr_auc\']:.3f}")\n    plt.xlabel("Recall")\n    plt.ylabel("Precision")\n    plt.title("Precision-Recall Curves")\n    plt.legend(fontsize=8)\n    plt.tight_layout()\n    plt.savefig(FIGURES / "pr_curves.png", dpi=220)\n    plt.close()\n\n    best_rf = fitted.get("Random Forest + Graph Features")\n    if best_rf is not None:\n        pre = best_rf.named_steps["preprocess"]\n        rf = best_rf.named_steps["model"]\n        names = pre.get_feature_names_out()\n        importances = pd.DataFrame({"feature": names, "importance": rf.feature_importances_})\n        importances = importances.sort_values("importance", ascending=False).head(20)\n        importances.to_csv(RESULTS / "feature_importance_random_forest.csv", index=False)\n        plt.figure(figsize=(9, 6))\n        sns.barplot(data=importances, y="feature", x="importance", color="#2E7D8F")\n        plt.title("Top Random Forest Feature Importances")\n        plt.ylabel("")\n        plt.tight_layout()\n        plt.savefig(FIGURES / "feature_importance.png", dpi=220)\n        plt.close()\n\n\ndef main() -> None:\n    ensure_dirs()\n    ensure_data()\n    print("Computing full-dataset statistics...")\n    stats = first_pass_stats()\n    plot_eda(stats)\n\n    print("Loading stratified model sample...")\n    sample = load_model_sample(stats)\n    sample = add_features(sample)\n    sample = add_graph_features(sample)\n    sample.to_parquet(RESULTS / "model_sample_features.parquet", index=False)\n    print(f"Model sample shape: {sample.shape}")\n    print(sample["Is Laundering"].value_counts(normalize=True).rename("rate"))\n\n    print("Running t-SNE and clustering...")\n    run_clustering_and_tsne(sample)\n\n    print("Training and evaluating supervised models...")\n    run_supervised_models(sample)\n    print("Done. Outputs written to figures/ and results/.")\n\n\nif __name__ == "__main__":\n    main()\n'
Path('src/aml_project_pipeline.py').write_text(PIPELINE_SOURCE, encoding='utf-8')
print('Pipeline source written to src/aml_project_pipeline.py')
# To regenerate all outputs from scratch, uncomment the next two lines.
# import runpy
# runpy.run_path('src/aml_project_pipeline.py', run_name='__main__')

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
figures = ROOT / "figures"
results = ROOT / "results"

metrics = pd.read_csv(results / "metrics_summary.csv")
clustering = pd.read_csv(results / "clustering_metrics.csv")
payment = pd.read_csv(results / "payment_format_rates.csv")
metrics

## Exploratory Data Analysis

In [ ]:
display(Image(filename=str(figures / "eda_payment_format_rate.png")))
display(Image(filename=str(figures / "eda_amount_distribution.png")))
display(Image(filename=str(figures / "eda_hourly_volume_rate.png")))

## t-SNE and Clustering

In [ ]:
clustering

In [ ]:
display(Image(filename=str(figures / "tsne_projection.png")))
display(Image(filename=str(figures / "clustering_kmeans_tsne.png")))
display(Image(filename=str(figures / "clustering_dbscan_tsne.png")))

## Supervised Learning Results

In [ ]:
metrics[["model", "precision", "recall", "f1", "roc_auc", "pr_auc"]]

In [ ]:
display(Image(filename=str(figures / "roc_curves.png")))
display(Image(filename=str(figures / "pr_curves.png")))
display(Image(filename=str(figures / "confusion_matrix_histgradientboosting_plus_graph_features.png")))
display(Image(filename=str(figures / "feature_importance.png")))

## Main Interpretation

The best model is HistGradientBoosting with graph-aware features. Compared with the base HistGradientBoosting model, graph features substantially improve F1 and PR-AUC. This supports the conclusion that AML detection benefits from modeling account behavior and transaction-network structure, not only individual transaction attributes.